# 05 - Weekly Summaries Data Wrangling

Notebook ini memproses dataset `weekly_summaries.csv`.

Tabel ini berisi ringkasan mingguan, sehingga pemeriksaan diarahkan pada validitas periode, konsistensi kategori, rentang agregasi numerik, dan keunikan kombinasi `user_id + week_start + week_end`.

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Mengatur tampilan dataframe agar output notebook lebih mudah dibaca.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Cari root project otomatis
# Mengambil lokasi kerja notebook saat ini.
current_path = Path.cwd().resolve()

# Menelusuri parent folder sampai menemukan root project yang memiliki folder data/raw.
for path in [current_path] + list(current_path.parents):
    if (path / "data" / "raw").exists():
        PROJECT_ROOT = path
        break

# Menentukan folder sumber data raw.
RAW_DIR = PROJECT_ROOT / "data" / "raw"
# Menentukan folder output data hasil cleaning.
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
# Menentukan folder output report dan validation summary.
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"

# Membuat folder processed jika belum tersedia.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
# Membuat folder reports jika belum tersedia.
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Menampilkan path project untuk memastikan notebook membaca folder yang benar.
print("PROJECT_ROOT :", PROJECT_ROOT)
print("RAW_DIR      :", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("REPORT_DIR   :", REPORT_DIR)

PROJECT_ROOT : C:\Data Codingan\student_stress_data_science
RAW_DIR      : C:\Data Codingan\student_stress_data_science\data\raw
PROCESSED_DIR: C:\Data Codingan\student_stress_data_science\data\processed
REPORT_DIR   : C:\Data Codingan\student_stress_data_science\outputs\reports


## 1. Load Dataset

In [ ]:
# memuat dataset dari folder yang sesuai dan menampilkan sampel awal data.
# Membaca file CSV ke dalam dataframe.
weekly_summaries = pd.read_csv(RAW_DIR / "weekly_summaries.csv")
users_clean = pd.read_csv(PROCESSED_DIR / "users_clean.csv")

# Menampilkan beberapa baris awal untuk memahami bentuk data.
weekly_summaries.head()

,id,user_id,week_start,week_end,average_stress_score,average_sleep_hours,average_screen_time,average_study_hours,high_stress_days,dominant_stress_level,stress_trend,main_trigger,created_at
0,1,1,2026-01-01,2026-01-07,52.92,7.10,7.23,3.46,0,Medium,Stable,Low Mood,2026-01-07 21:00:00
1,2,1,2026-01-08,2026-01-14,49.64,6.73,7.90,3.24,0,Medium,Stable,Academic Pressure,2026-01-14 21:00:00
2,3,1,2026-01-15,2026-01-21,50.77,7.53,8.34,3.56,0,Medium,Stable,Academic Pressure,2026-01-21 21:00:00
3,4,1,2026-01-22,2026-01-28,56.57,7.11,7.85,4.00,0,Medium,Decreasing,Low Mood,2026-01-28 21:00:00
4,5,1,2026-01-29,2026-02-04,67.72,6.63,8.76,4.89,3,Medium,Stable,Academic Pressure,2026-02-04 21:00:00


## 2. Assessing Data

In [ ]:
# menampilkan struktur dataframe, tipe data, dan jumlah nilai non-null.
# Menampilkan struktur kolom, tipe data, dan jumlah non-null.
weekly_summaries.info()

<class 'pandas.DataFrame'>
RangeIndex: 3600 entries, 0 to 3599
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     3600 non-null   int64  
 1   user_id                3600 non-null   int64  
 2   week_start             3600 non-null   str    
 3   week_end               3600 non-null   str    
 4   average_stress_score   3600 non-null   float64
 5   average_sleep_hours    3600 non-null   float64
 6   average_screen_time    3600 non-null   float64
 7   average_study_hours    3600 non-null   float64
 8   high_stress_days       3600 non-null   int64  
 9   dominant_stress_level  3600 non-null   str    
 10  stress_trend           3600 non-null   str    
 11  main_trigger           3600 non-null   str    
 12  created_at             3600 non-null   str    
dtypes: float64(4), int64(3), str(6)
memory usage: 365.8 KB


In [ ]:
# menampilkan ringkasan statistik numerik dan kategorikal.
# Menampilkan ringkasan statistik untuk kolom numerik dan kategorikal.
weekly_summaries.describe(include='all')

,id,user_id,week_start,week_end,average_stress_score,average_sleep_hours,average_screen_time,average_study_hours,high_stress_days,dominant_stress_level,stress_trend,main_trigger,created_at
count,3600.000000,3600.00000,3600,3600,3600.000000,3600.000000,3600.000000,3600.000000,3600.000000,3600,3600,3600,3600
unique,NaN,NaN,12,12,NaN,NaN,NaN,NaN,NaN,3,6,10,12
top,NaN,NaN,2026-01-01,2026-01-07,NaN,NaN,NaN,NaN,NaN,Medium,Stable,Academic Pressure,2026-01-07 21:00:00
freq,NaN,NaN,300,300,NaN,NaN,NaN,NaN,NaN,3379,2144,1560,300
mean,1800.500000,150.50000,NaN,NaN,56.214336,6.734550,6.728019,4.138792,0.715000,NaN,NaN,NaN,NaN
std,1039.374812,86.61409,NaN,NaN,6.768336,0.790324,1.440070,1.310616,1.318809,NaN,NaN,NaN,NaN
min,1.000000,1.00000,NaN,NaN,40.490000,3.380000,2.450000,0.500000,0.000000,NaN,NaN,NaN,NaN
25%,900.750000,75.75000,NaN,NaN,51.470000,6.220000,5.820000,3.190000,0.000000,NaN,NaN,NaN,NaN
50%,1800.500000,150.50000,NaN,NaN,54.995000,6.730000,6.700000,4.050000,0.000000,NaN,NaN,NaN,NaN
75%,2700.250000,225.25000,NaN,NaN,59.862500,7.250000,7.740000,5.060000,1.000000,NaN,NaN,NaN,NaN


In [ ]:
# menilai missing value, duplicate, dan kandidat masalah kualitas data.
print("Missing value:")
# Menghitung jumlah missing value pada setiap kolom.
print(weekly_summaries.isna().sum())

print("\nDuplicate user_id + week_start + week_end:")
# Mengecek keberadaan data duplicate berdasarkan aturan yang relevan.
print(weekly_summaries.duplicated(["user_id", "week_start", "week_end"]).sum())

print("\nStress trend unique:")
# Melihat variasi nilai unik untuk menilai konsistensi kategori atau format.
print(weekly_summaries["stress_trend"].unique())

Missing value:
id                       0
user_id                  0
week_start               0
week_end                 0
average_stress_score     0
average_sleep_hours      0
average_screen_time      0
average_study_hours      0
high_stress_days         0
dominant_stress_level    0
stress_trend             0
main_trigger             0
created_at               0
dtype: int64

Duplicate user_id + week_start + week_end:
0

Stress trend unique:
<StringArray>
['Stable', 'Decreasing', 'Increasing', 'increasing', 'decreasing', 'stable']
Length: 6, dtype: str


## Insight:

Tabel `weekly_summaries` memiliki karakter sebagai data agregasi. Kesalahan yang paling berisiko bukan pada format teks semata, tetapi pada periode mingguan yang tidak valid, duplicate weekly key, kategori trend yang tidak standar, atau nilai agregasi yang keluar dari batas logis.

Cleaning dilakukan dengan standardisasi tipe data, validasi user, validasi kategori `dominant_stress_level` dan `stress_trend`, pembatasan range agregasi numerik, serta penghapusan duplicate berdasarkan kombinasi periode mingguan.

## 3. Cleaning Data

Langkah cleaning:

1. Mengubah key, tanggal, dan kolom agregasi ke tipe data yang sesuai.
2. Menstandarkan kategori `dominant_stress_level` dan `stress_trend`.
3. Menghapus baris dengan informasi periode atau agregasi penting yang tidak valid.
4. Memastikan `user_id` tersedia pada `users_clean`.
5. Membatasi nilai agregasi numerik pada range yang logis.
6. Menghapus weekly summary dengan periode yang tidak valid.
7. Menghapus duplicate berdasarkan `user_id + week_start + week_end`.

In [ ]:
# membuat salinan dataframe lalu menjalankan proses cleaning sesuai hasil assessing.
weekly_summaries_clean = weekly_summaries.copy()

# Mengubah kolom ke tipe numerik; nilai yang gagal dikonversi menjadi NaN.
weekly_summaries_clean["id"] = pd.to_numeric(weekly_summaries_clean["id"], errors="coerce")
weekly_summaries_clean["user_id"] = pd.to_numeric(weekly_summaries_clean["user_id"], errors="coerce")
# Mengubah kolom ke tipe datetime; format yang tidak valid menjadi NaT.
weekly_summaries_clean["week_start"] = pd.to_datetime(weekly_summaries_clean["week_start"], errors="coerce")
weekly_summaries_clean["week_end"] = pd.to_datetime(weekly_summaries_clean["week_end"], errors="coerce")
# Mengubah kolom ke tipe numerik; nilai yang gagal dikonversi menjadi NaN.
weekly_summaries_clean["average_stress_score"] = pd.to_numeric(weekly_summaries_clean["average_stress_score"], errors="coerce")
weekly_summaries_clean["average_sleep_hours"] = pd.to_numeric(weekly_summaries_clean["average_sleep_hours"], errors="coerce")
weekly_summaries_clean["average_screen_time"] = pd.to_numeric(weekly_summaries_clean["average_screen_time"], errors="coerce")
weekly_summaries_clean["average_study_hours"] = pd.to_numeric(weekly_summaries_clean["average_study_hours"], errors="coerce")
weekly_summaries_clean["high_stress_days"] = pd.to_numeric(weekly_summaries_clean["high_stress_days"], errors="coerce")
# Membersihkan whitespace dan menstandarkan format teks.
weekly_summaries_clean["dominant_stress_level"] = weekly_summaries_clean["dominant_stress_level"].astype(str).str.strip().str.title()
weekly_summaries_clean["stress_trend"] = weekly_summaries_clean["stress_trend"].astype(str).str.strip().str.title()
weekly_summaries_clean["main_trigger"] = weekly_summaries_clean["main_trigger"].astype(str).str.strip()
# Mengubah kolom ke tipe datetime; format yang tidak valid menjadi NaT.
weekly_summaries_clean["created_at"] = pd.to_datetime(weekly_summaries_clean["created_at"], errors="coerce")

# Menghapus baris yang kehilangan kolom kunci atau informasi penting.
weekly_summaries_clean = weekly_summaries_clean.dropna(
    subset=[
        "id", "user_id", "week_start", "week_end",
        "average_stress_score", "high_stress_days",
        "dominant_stress_level", "stress_trend", "created_at"
    ]
)

valid_user_ids = set(users_clean["id"])
weekly_summaries_clean = weekly_summaries_clean[
    # Memvalidasi apakah nilai kolom berada dalam daftar nilai yang diperbolehkan.
    weekly_summaries_clean["user_id"].isin(valid_user_ids)
]

weekly_summaries_clean = weekly_summaries_clean[
    # Memvalidasi apakah nilai kolom berada dalam daftar nilai yang diperbolehkan.
    weekly_summaries_clean["dominant_stress_level"].isin(["Low", "Medium", "High"])
]

weekly_summaries_clean = weekly_summaries_clean[
    # Memvalidasi apakah nilai kolom berada dalam daftar nilai yang diperbolehkan.
    weekly_summaries_clean["stress_trend"].isin(["Increasing", "Stable", "Decreasing"])
]

# Membatasi nilai agar tetap berada pada rentang logis.
weekly_summaries_clean["average_stress_score"] = weekly_summaries_clean["average_stress_score"].clip(0, 100)
weekly_summaries_clean["average_sleep_hours"] = weekly_summaries_clean["average_sleep_hours"].clip(0, 24)
weekly_summaries_clean["average_screen_time"] = weekly_summaries_clean["average_screen_time"].clip(0, 24)
weekly_summaries_clean["average_study_hours"] = weekly_summaries_clean["average_study_hours"].clip(0, 24)
weekly_summaries_clean["high_stress_days"] = weekly_summaries_clean["high_stress_days"].clip(0, 7)

weekly_summaries_clean = weekly_summaries_clean[
    weekly_summaries_clean["week_end"] >= weekly_summaries_clean["week_start"]
]

# Mengurutkan data agar proses deduplikasi atau output lebih stabil.
weekly_summaries_clean = weekly_summaries_clean.sort_values("created_at")
# Menghapus duplicate sesuai subset key yang ditentukan.
weekly_summaries_clean = weekly_summaries_clean.drop_duplicates(
    subset=["user_id", "week_start", "week_end"],
    keep="last"
)

weekly_summaries_clean["id"] = weekly_summaries_clean["id"].astype(int)
weekly_summaries_clean["user_id"] = weekly_summaries_clean["user_id"].astype(int)
# Membulatkan nilai numerik agar format output lebih rapi.
weekly_summaries_clean["high_stress_days"] = weekly_summaries_clean["high_stress_days"].round().astype(int)
weekly_summaries_clean["week_start"] = weekly_summaries_clean["week_start"].dt.strftime("%Y-%m-%d")
weekly_summaries_clean["week_end"] = weekly_summaries_clean["week_end"].dt.strftime("%Y-%m-%d")
weekly_summaries_clean["created_at"] = weekly_summaries_clean["created_at"].dt.strftime("%Y-%m-%d %H:%M:%S")

weekly_summaries_clean = weekly_summaries_clean[
    [
        "id", "user_id", "week_start", "week_end",
        "average_stress_score", "average_sleep_hours", "average_screen_time",
        "average_study_hours", "high_stress_days", "dominant_stress_level",
        "stress_trend", "main_trigger", "created_at"
    ]
# Mengurutkan data agar proses deduplikasi atau output lebih stabil.
].sort_values("id")

# Menampilkan beberapa baris awal untuk memahami bentuk data.
weekly_summaries_clean.head()

,id,user_id,week_start,week_end,average_stress_score,average_sleep_hours,average_screen_time,average_study_hours,high_stress_days,dominant_stress_level,stress_trend,main_trigger,created_at
0,1,1,2026-01-01,2026-01-07,52.92,7.10,7.23,3.46,0,Medium,Stable,Low Mood,2026-01-07 21:00:00
1,2,1,2026-01-08,2026-01-14,49.64,6.73,7.90,3.24,0,Medium,Stable,Academic Pressure,2026-01-14 21:00:00
2,3,1,2026-01-15,2026-01-21,50.77,7.53,8.34,3.56,0,Medium,Stable,Academic Pressure,2026-01-21 21:00:00
3,4,1,2026-01-22,2026-01-28,56.57,7.11,7.85,4.00,0,Medium,Decreasing,Low Mood,2026-01-28 21:00:00
4,5,1,2026-01-29,2026-02-04,67.72,6.63,8.76,4.89,3,Medium,Stable,Academic Pressure,2026-02-04 21:00:00


## Insight Setelah Cleaning:

`weekly_summaries_clean` sudah memiliki periode mingguan yang valid, kategori yang standar, dan nilai agregasi yang berada dalam rentang logis. Dataset ini dapat digunakan untuk analisis tren mingguan dan kebutuhan dashboard monitoring berbasis ringkasan.

## 4. Validation dan Save Output

In [ ]:
# membuat tabel validasi untuk memastikan hasil cleaning memenuhi aturan kualitas data.
# Membuat dataframe validasi untuk mendokumentasikan hasil pengecekan kualitas data.
validation = pd.DataFrame([
    # Mengecek keberadaan data duplicate berdasarkan aturan yang relevan.
    {"rule": "weekly_summaries user_id + week_start + week_end unique", "passed": not weekly_summaries_clean.duplicated(["user_id", "week_start", "week_end"]).any()},
    # Memvalidasi apakah nilai kolom berada dalam daftar nilai yang diperbolehkan.
    {"rule": "stress_trend valid", "passed": weekly_summaries_clean["stress_trend"].isin(["Increasing", "Stable", "Decreasing"]).all()},
    {"rule": "dominant_stress_level valid", "passed": weekly_summaries_clean["dominant_stress_level"].isin(["Low", "Medium", "High"]).all()},
])

validation

,rule,passed
0,weekly_summaries user_id + week_start + week_e...,True
1,stress_trend valid,True
2,dominant_stress_level valid,True


In [ ]:
# menyimpan output hasil cleaning atau report ke folder tujuan.
# Menyimpan dataframe ke file CSV.
weekly_summaries_clean.to_csv(PROCESSED_DIR / "weekly_summaries_clean.csv", index=False)
validation.to_csv(REPORT_DIR / "weekly_summaries_validation.csv", index=False)

print("Saved:", PROCESSED_DIR / "weekly_summaries_clean.csv")

Saved: C:\Data Codingan\student_stress_data_science\data\processed\weekly_summaries_clean.csv
